# 🤖 Autonomous Local Data Analyst — Dual-Model Edition
### Reasoning: `phi4-mini-reasoning` (Microsoft) · Coding: `qwen2.5-coder:7b-instruct` (Alibaba)

**Why two models?**
- **phi4-mini-reasoning** — 3.8B, fits in ~2.5GB VRAM, native `<think>` CoT, best reasoning at 8GB tier. Handles CLASSIFY, PLAN, CHECK, EXPLAIN.
- **qwen2.5-coder:7b-instruct** — 7B, ~4.7GB VRAM Q4_K_M, 88.4% HumanEval. Handles CODE only.
- Combined they use ~7.2GB — both fit in your 8GB VRAM simultaneously (Ollama keeps both hot).

**Prompt formats used:**
- `phi4-mini-reasoning`: ChatML with `system` role + `<think>` directive in the user turn. The model emits `<think>...</think>` before its answer; we strip thinking tokens from non-code replies.
- `qwen2.5-coder:7b-instruct`: Standard ChatML (`system` / `user` / `assistant`). System role works correctly. Responds well to explicit output contracts and code-only directives.

**Licenses:** Both Apache 2.0. Both open-weight. Microsoft + Alibaba.


In [ ]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
import subprocess, sys

packages = ['gradio', 'ollama', 'pandas', 'openpyxl', 'xlrd', 'matplotlib', 'seaborn', 'tabulate']
for pkg in packages:
    print(f'Installing {pkg}...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
print('\n✅ All packages installed!')


In [ ]:
# ── Cell 2: Pull both models ───────────────────────────────────────────────────
import subprocess

REASONING_MODEL = 'phi4-mini-reasoning'   # Classifier, Planner, Checker, Explainer
CODING_MODEL    = 'qwen2.5-coder:7b-instruct'  # Code generation only

for model in [REASONING_MODEL, CODING_MODEL]:
    print(f'Pulling {model}...')
    result = subprocess.run(['ollama', 'pull', model], capture_output=False)
    if result.returncode == 0:
        print(f'  ✅ {model} ready!')
    else:
        print(f'  ⚠️  Pull may have failed for {model}')


In [ ]:
# ── Cell 3: Imports ────────────────────────────────────────────────────────────
import os, glob, json, re, io, base64, traceback, sys, tempfile
from pathlib import Path
from io import StringIO

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import gradio as gr
import ollama

print(f'✅ Imports OK  |  Gradio {gr.__version__}')


In [ ]:
# ── Cell 4: Config ─────────────────────────────────────────────────────────────
REASONING_MODEL  = 'phi4-mini-reasoning'        # All reasoning phases
CODING_MODEL     = 'qwen2.5-coder:7b-instruct'  # Code generation only

MAX_ROWS_CTX = 30
MAX_COLS_CTX = 20

# phi4-mini-reasoning: 3.8B → small ctx is fine; 8192 leaves plenty of headroom
REASONING_CTX = 8192
# qwen2.5-coder:7b supports 128k; we give it 12k for full schema + history
CODING_CTX    = 12288

DATAFRAMES   = {}
DATA_CONTEXT = ''

SUPPORTED_EXTENSIONS = ('.csv', '.xlsx', '.xlsm', '.xlsb', '.xls', '.tsv')

print(f'✅ Config ready.')
print(f'   Reasoning model : {REASONING_MODEL}  (ctx={REASONING_CTX})')
print(f'   Coding model    : {CODING_MODEL}  (ctx={CODING_CTX})')


In [ ]:
# ── Cell 5: Data loading helpers ───────────────────────────────────────────────

def _try_read_csv(path):
    encodings = ['utf-8-sig', 'utf-8', 'cp1252', 'latin1']
    separators = [',', ';', '\t', '|']
    last_err = None
    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc, low_memory=False,
                             on_bad_lines='skip', engine='python', sep=None)
            if df.shape[1] > 1 or len(df) > 0:
                return df
        except Exception as e:
            last_err = e
        for sep in separators:
            try:
                df = pd.read_csv(path, encoding=enc, sep=sep, low_memory=False,
                                 on_bad_lines='skip', engine='c')
                if df.shape[1] > 1 or len(df) > 0:
                    return df
            except Exception as e:
                last_err = e
    raise ValueError(f'Could not parse CSV — last error: {last_err}')


def load_file(tmp_path, orig_name):
    ext = Path(orig_name).suffix.lower()
    try:
        if ext in ('.csv', '.tsv'):
            return _try_read_csv(tmp_path)
        if ext in ('.xlsx', '.xlsm', '.xlsb'):
            engine = 'openpyxl'
        elif ext == '.xls':
            engine = 'xlrd'
        else:
            return None
        sheets = pd.read_excel(tmp_path, sheet_name=None, engine=engine)
        if len(sheets) == 1:
            return next(iter(sheets.values()))
        return sheets
    except Exception as e:
        print(f'  ⚠️  Could not load {orig_name}: {e}')
        return None


def profile_dataframe(name, df):
    lines = [f'### Table: `{name}`',
             f'- Rows: {len(df):,}  |  Columns: {len(df.columns)}']
    for col in df.columns[:MAX_COLS_CTX]:
        dtype  = str(df[col].dtype)
        n_null = int(df[col].isna().sum())
        n_uniq = int(df[col].nunique())
        if pd.api.types.is_numeric_dtype(df[col]):
            extra = f'  range=[{df[col].min()}, {df[col].max()}]'
        else:
            top   = df[col].value_counts().head(3).index.tolist()
            extra = f'  top={top}'
        lines.append(f'  - {col} ({dtype}) | nulls={n_null} | unique={n_uniq}{extra}')
    if len(df.columns) > MAX_COLS_CTX:
        lines.append(f'  ... and {len(df.columns) - MAX_COLS_CTX} more columns')
    sample = df.head(MAX_ROWS_CTX).to_string(index=False, max_cols=MAX_COLS_CTX)
    lines += [f'\nSample rows (first {min(MAX_ROWS_CTX, len(df))}):', '```', sample, '```']
    return '\n'.join(lines)


def _finalize(dfs):
    global DATAFRAMES, DATA_CONTEXT
    DATAFRAMES   = dfs
    DATA_CONTEXT = '\n\n'.join(profile_dataframe(n, df) for n, df in dfs.items())


def _resolve_file_entry(f):
    if isinstance(f, dict):
        orig = Path(f.get('name', f.get('orig_name', ''))).name
        tmp  = f.get('path', f.get('tmp_path', str(f.get('name', ''))))
        return str(tmp), orig
    tmp = str(f)
    orig = Path(f.orig_name).name if hasattr(f, 'orig_name') and f.orig_name else Path(tmp).name
    return tmp, orig


def load_files(file_list, source_label='selected files'):
    entries = [_resolve_file_entry(f) for f in file_list]
    valid = [(p, n) for p, n in entries
             if Path(n).suffix.lower() in SUPPORTED_EXTENSIONS and os.path.isfile(p)]
    if not valid:
        return {}, '❌ No supported CSV/Excel files found.'
    dfs, msgs = {}, [f'📂 Loading {len(valid)} file(s)...']
    for tmp_path, orig_name in sorted(valid, key=lambda x: x[1]):
        result = load_file(tmp_path, orig_name)
        if result is None:
            msgs.append(f'  ⚠️  {orig_name} — skipped')
        elif isinstance(result, dict):
            base = Path(orig_name).stem
            for sheet_name, df in result.items():
                key = f'{base} [{sheet_name}]'
                dfs[key] = df
                msgs.append(f'  ✅ {key}  ({len(df):,} rows × {len(df.columns)} cols)')
        else:
            dfs[orig_name] = result
            msgs.append(f'  ✅ {orig_name}  ({len(result):,} rows × {len(result.columns)} cols)')
    _finalize(dfs)
    msgs.append(f'\n✅ **{len(dfs)} table(s) loaded and ready!**')
    return dfs, '\n'.join(msgs)


def load_folder(folder_path):
    if not folder_path or not os.path.isdir(folder_path):
        return {}, '❌ Invalid folder path.'
    fps = []
    for ext in SUPPORTED_EXTENSIONS:
        fps.extend(glob.glob(os.path.join(folder_path, f'*{ext}')))
    if not fps:
        return {}, f'❌ No CSV/Excel files found in `{folder_path}`.'
    return load_files([str(p) for p in fps], source_label=f'`{folder_path}`')


print('✅ Data loading helpers defined')


In [ ]:
# ── Cell 6: Pandas execution sandbox ──────────────────────────────────────────

import io, base64, traceback, sys, re
from pathlib import Path
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

try:
    from tabulate import tabulate as _tabulate
    _HAS_TABULATE = True
except ImportError:
    _HAS_TABULATE = False


def execute_pandas_code(code):
    exec_env = {'__builtins__': __builtins__, 'pd': pd, 'np': np, 'plt': plt, 'sns': sns}
    if _HAS_TABULATE:
        exec_env['tabulate'] = _tabulate

    for fname, df in DATAFRAMES.items():
        stem  = Path(fname).stem
        clean = re.sub(r'[^a-zA-Z0-9_]', '_', stem)
        short = clean.replace('olist_', '').replace('_dataset', '')
        for key in [fname, stem, clean, short]:
            exec_env[key] = df

    old_stdout = sys.stdout
    sys.stdout = buf = StringIO()
    image_b64  = None
    out_text   = ''

    try:
        exec(code, exec_env)
        sys.stdout = old_stdout
        out_text   = buf.getvalue()

        if plt.get_fignums():
            img_buf = io.BytesIO()
            plt.savefig(img_buf, format='png', bbox_inches='tight', dpi=150)
            img_buf.seek(0)
            image_b64 = base64.b64encode(img_buf.read()).decode()
            plt.close('all')

        if not out_text.strip():
            for vname in ['result', 'results', 'output', 'answer', 'final',
                          'top5', 'top_5', 'top10', 'top_10', 'summary',
                          'df_result', 'result_df', 'monthly', 'pivot',
                          'grouped', 'agg', 'counts', 'stats']:
                if vname in exec_env:
                    r = exec_env[vname]
                    out_text = r.to_string() if isinstance(r, (pd.DataFrame, pd.Series)) else str(r)
                    if out_text.strip():
                        break

    except Exception:
        sys.stdout = old_stdout
        err_text = traceback.format_exc()
        col_listing = '\nAvailable columns per table:\n' + '\n'.join(
            f'  {n}: {list(df.columns)}' for n, df in DATAFRAMES.items()
        )
        out_text = 'ERROR:\n' + err_text + col_listing
        plt.close('all')

    return out_text.strip(), image_b64


print('✅ Sandbox defined')


In [ ]:
# ── Cell 7: Dual-Model Adaptive Pipeline ──────────────────────────────────────
#
# MODEL ROUTING:
#   REASONING_MODEL (phi4-mini-reasoning) → CLASSIFY, PLAN, CHECK, EXPLAIN
#   CODING_MODEL    (qwen2.5-coder:7b)    → CODE GEN + FIX only
#
# PROMPT FORMAT — phi4-mini-reasoning:
#   Uses ChatML. Prompt the model to reason inside <think>...</think>.
#   We strip <think> blocks from final output so they don't pollute results.
#   Temperature 0.0 for CLASSIFY/CHECK (deterministic), 0.3 for PLAN/EXPLAIN.
#
# PROMPT FORMAT — qwen2.5-coder:7b-instruct:
#   Full ChatML with system role (works correctly on this model).
#   Explicit "Output ONLY one ```python``` block" contract + Defensive Coding.
#   Temperature 0.1 for code gen (low variance, high correctness).

import re, json as _json
from pathlib import Path
import ollama

# ─── Helpers ──────────────────────────────────────────────────────────────────

def _build_schema():
    if not DATAFRAMES:
        return ''
    lines = []
    for fname, df in DATAFRAMES.items():
        stem  = Path(fname).stem
        clean = re.sub(r'[^a-zA-Z0-9_]', '_', stem)
        short = clean.replace('olist_', '').replace('_dataset', '')
        aliases = list(dict.fromkeys([clean, short]))
        others = ', '.join(aliases[1:])
        alias_str = f' (also: {others})' if others else ''
        col_info = ', '.join(f'"{c}"({str(dt)})' for c, dt in df.dtypes.items())
        lines.append(f'`{clean}`{alias_str}: {col_info}')
    return '\n'.join(lines)


def _build_table_names():
    lines = []
    for fname in DATAFRAMES:
        stem  = Path(fname).stem
        clean = re.sub(r'[^a-zA-Z0-9_]', '_', stem)
        short = clean.replace('olist_', '').replace('_dataset', '')
        aliases = list(dict.fromkeys([clean, short]))
        lines.append(', '.join(f'`{a}`' for a in aliases))
    return '\n'.join(lines)


def _strip_think(text):
    # Remove <think>...</think> blocks from phi4-mini-reasoning output.
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()


def _llm_reason(messages, temperature=0.0):
    # Call phi4-mini-reasoning.
    # - Supports system role natively via ChatML.
    # - Strips <think> blocks from the returned text.
    response = ollama.chat(
        model=REASONING_MODEL,
        messages=messages,
        options={
            'temperature': temperature,
            'num_ctx': REASONING_CTX,
            'top_p': 0.9,
        },
    )
    return _strip_think(response['message']['content'])


def _llm_code(messages, temperature=0.1):
    # Call qwen2.5-coder:7b-instruct.
    # - Full ChatML with system role.
    # - Higher ctx for long schema + history.
    # - Do NOT strip think tokens (this model doesn't emit them).
    response = ollama.chat(
        model=CODING_MODEL,
        messages=messages,
        options={
            'temperature': temperature,
            'num_ctx': CODING_CTX,
            'top_p': 0.95,
            'repeat_penalty': 1.05,
        },
    )
    return response['message']['content']


def _sanitize_content(c):
    if isinstance(c, list):
        return ' '.join(p.get('text', str(p)) for p in c if isinstance(p, dict))
    return str(c) if c is not None else ''


def _parse_json(raw):
    text = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    text = re.sub(r'^```json\s*', '', text, flags=re.IGNORECASE)
    text = re.sub(r'^```\s*', '', text)
    text = re.sub(r'\s*```$', '', text)
    try:
        return _json.loads(text)
    except Exception:
        m = re.search(r'\{.*\}', text, re.DOTALL)
        if m:
            try:
                return _json.loads(m.group(0))
            except Exception:
                pass
    return None


# ─── Prompts ──────────────────────────────────────────────────────────────────

CLASSIFY_SYSTEM = (
    "You are a query classifier for a pandas data analyst pipeline.\n"
    "You must determine if a query is SIMPLE or COMPLEX.\n\n"
    "Mark COMPLEX if ANY of these apply:\n"
    "1. Requires 3+ tables joined together.\n"
    "2. Requires a lookup table to decode IDs into names.\n"
    "3. Requires two-level aggregation (e.g., sum per order THEN avg per group).\n"
    "4. Requires a multi-hop join chain to reach target attributes.\n"
    "Otherwise, mark SIMPLE.\n\n"
    "OUTPUT FORMAT:\n"
    "First, output your reasoning inside <think>...</think> tags.\n"
    "Then, output EXACTLY ONE WORD (either SIMPLE or COMPLEX) and nothing else."
)

PLAN_SYSTEM = (
    "You are a senior data analyst planning a pandas workflow.\n"
    "Schema: {schema}\n\n"
    "OUTPUT FORMAT:\n"
    "1. First, use <think>...</think> tags to identify necessary tables, required joins, and aggregation steps.\n"
    "2. Next, output a STRICTly formatted JSON block inside ```json fences.\n\n"
    "JSON SCHEMA TO FOLLOW:\n"
    "{{\n"
    "  \"tables_needed\": [\"exact_var_name\"],\n"
    "  \"join_steps\": [\"merge X with Y on col\"],\n"
    "  \"aggregation\": \"precise description\",\n"
    "  \"visualization\": null\n"
    "}}"
)

# Simplified user templates since the System prompts now strictly handle formatting
CLASSIFY_USER_TMPL  = '{question}'
PLAN_USER_TMPL      = '{question}'
CHECK_USER_TMPL     = 'Verify.\n\n<think>Does every part of the question appear in the output?</think>'

CODE_SYSTEM = (
    "You are an expert Python/pandas data engineer.\n\n"
    "Pre-loaded DataFrames (Do NOT call pd.read_csv):\n"
    "{table_names}\n\n"
    "Column schema:\n"
    "{schema}\n\n"
    "CRITICAL RULES:\n"
    "1. Output EXACTLY ONE ```python``` block. Do not explain your code.\n"
    "2. Assume `pandas as pd` and `matplotlib.pyplot as plt` are already imported.\n"
    "3. DEFENSIVE CODING: Handle missing values (`fillna` or `dropna`) before aggregations. Ensure data types match before merges.\n"
    "4. STRING SPLITTING: If splitting comma-separated text, ALWAYS use `.str.strip()` to remove extra whitespaces before grouping.\n"
    "5. TEXT OUTPUT: Always print final results as Markdown using `print(table_name.to_markdown(index=False))` so the reasoning model can read it easily.\n"
    "6. CHARTS: Create the plt object, apply `plt.tight_layout()` and `plt.xticks(rotation=45, ha='right')`. DO NOT call `plt.show()`. You MUST also `print(table_name.to_markdown(index=False))` for the underlying data powering the chart.\n"
    "7. NEVER invent column names. Only use the schema provided.\n"
    "8. VARIABLE NAMES: NEVER use a generic 'df' variable. You MUST use the exact table names provided in the Pre-loaded DataFrames section."
)

FIX_PROMPT_TMPL = (
    "Your previous code failed with this exact error:\n"
    "{error}\n\n"
    "Original Code that caused the error:\n"
    "```python\n"
    "{code}\n"
    "```\n\n"
    "Available columns to verify your fix:\n"
    "{schema}\n\n"
    "Diagnose the issue based on the error message. Output ONLY the corrected ```python``` block. Do not explain."
)

EMPTY_PROMPT = (
    "The code ran without error but printed nothing. "
    "Add print() on the final result. Output ONLY the corrected ```python``` block."
)

EXPLAIN_SYSTEM = (
    "You are a data analyst briefing a business executive.\n\n"
    "Rules:\n"
    "1. Quote every number exactly as it appears in the data output — never round or paraphrase.\n"
    "2. CONTEXT: Do not assume meanings for any short forms or abbreviations (e.g., geographic codes, acronyms) unless explicitly stated. Infer context strictly from the provided data.\n"
    "3. State the single most important business implication in 3-5 sentences.\n"
    "4. Use plain prose only — no bullet points, headers, or markdown.\n"
    "5. DO NOT invent or hallucinate any figures. If the data output is empty, explicitly state that no records met the criteria.\n\n"
    "First, <think> about the most critical metric in the data.</think> Then, provide the executive briefing."
)

CHECK_SYSTEM = (
    "Question: {question}\n"
    "Output: {output}\n"
    "Does this fully answer the question? Reply PASS or: RETRY: [one sentence on what is missing]"
)


# ─── Main pipeline ─────────────────────────────────────────────────────────────

def chat_with_analyst(message, history):
    # No data loaded — plain chat with reasoning model
    if not DATAFRAMES:
        msgs = [{'role': 'system', 'content': 'You are a helpful AI assistant.'},
                {'role': 'user',   'content': str(message)}]
        try:
            return _llm_reason(msgs, temperature=0.3), None
        except Exception as e:
            return f'Ollama error: {e}', None

    schema    = _build_schema()
    tbl_names = _build_table_names()

    # ── STEP 0: CLASSIFY (reasoning model) ────────────────────────────────────
    complexity = 'SIMPLE'
    try:
        cls_reply = _llm_reason([
            {'role': 'system', 'content': CLASSIFY_SYSTEM},
            {'role': 'user',   'content': CLASSIFY_USER_TMPL.format(question=message)
                                          + '\n\nAvailable tables:\n' + schema},
        ], temperature=0.0)
        if 'COMPLEX' in cls_reply.upper():
            complexity = 'COMPLEX'
    except Exception:
        complexity = 'SIMPLE'

    # ── STEP 1: PLAN (reasoning model, COMPLEX only) ───────────────────────────
    plan_context = ''
    if complexity == 'COMPLEX':
        try:
            plan_raw = _llm_reason([
                {'role': 'system', 'content': PLAN_SYSTEM.format(schema=schema)},
                {'role': 'user',   'content': PLAN_USER_TMPL.format(question=message)},
            ], temperature=0.3)
            plan_obj = _parse_json(plan_raw)
            if plan_obj:
                plan_context = '\n\nAnalysis plan:\n' + _json.dumps(plan_obj, indent=2)
        except Exception:
            pass

    # ── STEP 2+3: CODE + EXECUTE (coding model, self-correcting) ──────────────
    MAX_RETRIES = 3
    exec_out    = ''
    image_b64   = None
    last_error  = ''
    last_blocks = []

    user_msg = message + plan_context

    code_messages = [
        {'role': 'system', 'content': CODE_SYSTEM.format(
            table_names=tbl_names, schema=schema)},
    ]
    for turn in history:
        if isinstance(turn, dict):
            c = _sanitize_content(turn.get('content', ''))
            if turn['role'] == 'assistant' and 'Could not produce' in c:
                continue
            code_messages.append({'role': turn['role'], 'content': c})
    code_messages.append({'role': 'user', 'content': user_msg})

    for attempt in range(MAX_RETRIES):
        try:
            llm_reply = _llm_code(code_messages, temperature=0.1)
        except Exception as e:
            return f'Ollama error: {e}', None

        blocks = re.findall(r'```python(.*?)```', llm_reply, re.DOTALL)
        if not blocks:
            code_messages.append({'role': 'assistant', 'content': llm_reply})
            code_messages.append({'role': 'user',
                'content': 'No ```python``` block found. Output ONLY the ```python``` block.'})
            continue

        last_blocks = blocks
        exec_out   = ''
        image_b64  = None
        last_error = ''

        for blk in blocks:
            out, img = execute_pandas_code(blk.strip())
            if img:
                image_b64 = img
            if out.startswith('ERROR:'):
                last_error = out
            else:
                exec_out = (exec_out + '\n' + out).strip()

        if last_error:
            code_messages.append({'role': 'assistant', 'content': llm_reply})
            code_messages.append({'role': 'user', 'content': FIX_PROMPT_TMPL.format(
                code=blocks[0].strip(), error=last_error, schema=schema)})
            continue

        if not exec_out.strip() and not image_b64:
            code_messages.append({'role': 'assistant', 'content': llm_reply})
            code_messages.append({'role': 'user', 'content': EMPTY_PROMPT})
            continue

        break
    else:
        return (
            f'⚠️ Could not produce working code after {MAX_RETRIES} attempts.\n\n'
            f'Last error:\n```\n{last_error or exec_out}\n```',
            None,
        )

    # ── STEP 4: CHECK (reasoning model) ───────────────────────────────────────
    if exec_out.strip():
        try:
            check_reply = _llm_reason([
                {'role': 'system', 'content': CHECK_SYSTEM.format(
                    question=message, output=exec_out.strip())},
                {'role': 'user', 'content': CHECK_USER_TMPL},
            ], temperature=0.0)
            if check_reply.strip().upper().startswith('RETRY'):
                fix_instruction = check_reply.strip()[6:].strip()
                code_messages.append({'role': 'assistant', 'content': llm_reply})
                code_messages.append({'role': 'user', 'content':
                    f'Output is incomplete: {fix_instruction}\n'
                    f'Original code:\n```python\n{last_blocks[0].strip()}\n```\n'
                    f'Fix it. Output ONLY the corrected ```python``` block.'})
                try:
                    r2 = _llm_code(code_messages, temperature=0.1)
                    b2 = re.findall(r'```python(.*?)```', r2, re.DOTALL)
                    if b2:
                        o2, i2 = execute_pandas_code(b2[0].strip())
                        if i2:
                            image_b64 = i2
                        if o2 and not o2.startswith('ERROR:'):
                            exec_out = o2.strip()
                except Exception:
                    pass
        except Exception:
            pass

    # ── STEP 5: EXPLAIN (reasoning model) ─────────────────────────────────────
    explain_input = (exec_out.strip() if exec_out.strip()
                     else '[Chart generated — describe visible trends only, no invented numbers.]')
    try:
        explanation = _llm_reason([
            {'role': 'system', 'content': EXPLAIN_SYSTEM},
            {'role': 'user',   'content':
                f'Question: {message}\nData output:\n```\n{explain_input}\n```\n'
                f'Provide analysis.'},
        ], temperature=0.3)
    except Exception as e:
        explanation = f'(Explanation unavailable: {e})'

    if exec_out.strip():
        final_reply = f'**Output:**\n```\n{exec_out.strip()}\n```\n\n**Analysis:**\n{explanation.strip()}'
    else:
        final_reply = explanation

    return final_reply, image_b64


print('✅ Dual-model pipeline defined')
print(f'   CLASSIFY  → {REASONING_MODEL}')
print(f'   PLAN      → {REASONING_MODEL}')
print(f'   CODE      → {CODING_MODEL}')
print(f'   FIX       → {CODING_MODEL}')
print(f'   CHECK     → {REASONING_MODEL}')
print(f'   EXPLAIN   → {REASONING_MODEL}')

In [ ]:
# ── Cell 8: Gradio UI ──────────────────────────────────────────────────────────

CSS = """
#app-title {
  background: linear-gradient(90deg, #6d28d9, #0369a1);
  -webkit-background-clip: text;
  -webkit-text-fill-color: transparent;
  font-weight: 700;
  text-align: center;
}
#chatbot { min-height: 650px; }
"""


def on_load_uploaded(uploaded_files):
    if not uploaded_files:
        return '⚠️ Select a folder first.'
    if not isinstance(uploaded_files, list):
        uploaded_files = [uploaded_files]
    dfs, _ = load_files(uploaded_files, source_label='uploaded folder')
    if dfs:
        return f'✅ **Ready!** Loaded {len(dfs)} table(s).'
    return '❌ Failed to load files. Check format.'


def on_load_folder_path(folder_path):
    if not folder_path or not folder_path.strip():
        return '⚠️ Enter a folder path first.'
    dfs, _ = load_folder(folder_path.strip())
    if dfs:
        return f'✅ **Ready!** Loaded {len(dfs)} table(s).'
    return '❌ Failed to load files. Check path.'


def get_table_summary():
    if not DATAFRAMES:
        return '_No data loaded yet._'
    rows = ['| Table | Rows | Cols |', '|---|---|---|']
    for name, df in DATAFRAMES.items():
        rows.append(f'| `{name}` | {len(df):,} | {len(df.columns)} |')
    return '\n'.join(rows)


def respond(message, history):
    if not message or not message.strip():
        yield history, None, ''
        return
    reply, image_b64 = chat_with_analyst(message, history)
    history = list(history) + [
        {'role': 'user',      'content': message},
        {'role': 'assistant', 'content': reply},
    ]
    img_path = None
    if image_b64:
        tmp = tempfile.NamedTemporaryFile(suffix='.png', delete=False)
        tmp.write(base64.b64decode(image_b64))
        tmp.close()
        img_path = tmp.name
    yield history, img_path, ''


with gr.Blocks(title='Autonomous Data Analyst — Dual Model', css=CSS) as demo:

    gr.Markdown(
        '<h1 id=\'app-title\'>✦ Autonomous Data Analyst</h1>'
        '<p style=\'text-align:center;color:#6b7280;margin-top:-8px\'>'
        '🧠 Reasoning: <code>phi4-mini-reasoning</code> &nbsp;|&nbsp; '
        '💻 Coding: <code>qwen2.5-coder:7b</code></p>'
    )

    with gr.Accordion('📎 Attach Data', open=False):
        with gr.Row():
            folder_upload   = gr.File(label='Upload Folder', file_count='directory',
                                      type='filepath', height=100)
            folder_path_box = gr.Textbox(label='Or enter local folder path',
                                         placeholder='C:/Users/You/data')
        with gr.Row():
            load_btn      = gr.Button('⚡ Load Uploaded', variant='secondary')
            load_path_btn = gr.Button('⚡ Load Path',     variant='secondary')

    status_box = gr.Markdown('')

    with gr.Row():
        with gr.Column(scale=4):
            chatbot = gr.Chatbot(label='', elem_id='chatbot', height=650,
                                 show_label=False,
                                 avatar_images=(None, 'https://upload.wikimedia.org/wikipedia/commons/0/04/ChatGPT_logo.svg'))
            with gr.Row():
                msg_input = gr.Textbox(placeholder='Ask about your data…',
                                       lines=1, scale=6, show_label=False)
                send_btn  = gr.Button('Send 🚀', variant='primary', scale=1)
                clear_btn = gr.Button('🗑️ Clear', scale=1)

        with gr.Column(scale=1):
            chart_output = gr.Image(label='📈 Chart', type='filepath')
            with gr.Accordion('📋 Tables', open=True):
                table_summary = gr.Markdown(get_table_summary)

    load_btn.click(fn=on_load_uploaded, inputs=folder_upload,
                   outputs=status_box).then(fn=get_table_summary, outputs=table_summary)
    load_path_btn.click(fn=on_load_folder_path, inputs=folder_path_box,
                        outputs=status_box).then(fn=get_table_summary, outputs=table_summary)
    send_btn.click(fn=respond, inputs=[msg_input, chatbot],
                   outputs=[chatbot, chart_output, msg_input])
    msg_input.submit(fn=respond, inputs=[msg_input, chatbot],
                     outputs=[chatbot, chart_output, msg_input])
    clear_btn.click(fn=lambda: ([], None, ''), outputs=[chatbot, chart_output, msg_input])

print('✅ Gradio UI defined')


In [ ]:
# ── Cell 9: Launch ─────────────────────────────────────────────────────────────

try:
    listed      = ollama.list().get('models', [])
    model_names = [m.get('name') or m.get('model', '') for m in listed]
    for model in [REASONING_MODEL, CODING_MODEL]:
        found = any(model in m for m in model_names)
        status = '✅' if found else '⚠️  NOT FOUND — run: ollama pull ' + model
        print(f'{status}  {model}')
except Exception as e:
    print(f'❌ Cannot reach Ollama: {e}')
    print('   Run `ollama serve` in a terminal first.')

launched = False
for port in range(7860, 7871):
    try:
        print(f'\n🚀 Launching on port {port}...')
        demo.launch(server_name='0.0.0.0', server_port=port,
                    share=False, inbrowser=True)
        launched = True
        break
    except OSError:
        print(f'  Port {port} busy, trying next...')

if not launched:
    print('❌ No free port in range 7860-7870.')
